# WeaveGuard AI — Étape E : recherche de textiles similaires (ResNet-50 + FAISS)
Sortie du moteur : **« Textiles similaires »**, avec une similarité cosinus (plus élevée = plus proche). Ce n'est **ni une détection de défaut, ni une probabilité** (cahier des charges §7.3).

Deux galeries de référence, deux index :
- **DTD** (47 catégories de textures, 5640 images) : galerie générique, évaluée avec le **split officiel 1** (galerie = train+val, requêtes = test).
- **WeaveGuard v2** (références du domaine : images train+val, avec leurs classes de défauts) : évaluée avec train = galerie, val = requêtes. Le v2 ayant été regroupé par quasi-doublons, une requête n'a pas de copie proche dans la galerie.

Sorties versionnées sur Drive : `models/similarity/similarity_v1/` (index FAISS, métadonnées, configuration, fiche modèle, hash).

In [ ]:
# 1) Drive + dépendances + GPU
from google.colab import drive
drive.mount('/content/drive')
!pip -q install faiss-cpu
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (le calcul sera plus lent)')

In [ ]:
# 2) Configuration
from pathlib import Path
import json, hashlib, zipfile, datetime, time, random, collections
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import faiss
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

WG = Path('/content/drive/MyDrive/WeaveGuard')
DRIVE_OUT = WG / 'datasets'
V2 = Path('/content/weaveguard_dataset_v2')
OUT = WG / 'models' / 'similarity' / 'similarity_v1'; OUT.mkdir(parents=True, exist_ok=True)
DTD_DIR = next((p for p in [WG / 'data' / 'dtd', WG / 'data'] if (p / 'images').exists() and (p / 'labels').exists()), None)
assert DTD_DIR is not None, 'DTD introuvable (attendu : WeaveGuard/data/dtd/{images,labels})'
SEED = 42; dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH = 64; N_THREADS = 16; K = 10
EXTRACTOR_ID = 'resnet50_imagenet1k_v1_gap2048_l2'
ALLOWED_FIELDS_DTD = ['ref_id', 'source', 'category']
ALLOWED_FIELDS_WG = ['ref_id', 'source', 'classes', 'split']
print('DTD :', DTD_DIR, '| device :', dev)

In [ ]:
# 3) Mesures de recherche (sans PyTorch)
import numpy as np

def retrieval_metrics(Q, G, qlab, glab, ks=(1, 5, 10)):
    S = Q @ G.T; order = np.argsort(-S, axis=1)
    rel = (glab[order] == qlab[:, None])
    out = {f'P@{k}': float(rel[:, :k].mean()) for k in ks}
    prec = np.cumsum(rel, axis=1) / np.arange(1, rel.shape[1] + 1)[None]
    nrel = rel.sum(1); ap = (prec * rel).sum(1) / np.maximum(nrel, 1)
    out['mAP'] = float(ap[nrel > 0].mean())
    return out, order, rel

def multilabel_hits(Pq, Pg, order, ks=(1, 5)):
    share = (Pq @ Pg.T) > 0                                   # la requête et la référence partagent au moins une classe
    out = {f'hit@{k}': float(np.take_along_axis(share, order[:, :k], 1).any(1).mean()) for k in ks}
    out['chance_hit@1'] = float(share.mean())                 # probabilité qu'une référence tirée au hasard partage une classe
    return out

In [ ]:
# 4) Extracteur ResNet-50 (pooling global, vecteur 2048 normalisé L2)
class Embedder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet50(weights='IMAGENET1K_V1')
        self.body = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2, m.layer3, m.layer4)
    @torch.no_grad()
    def forward(self, x):
        return F.normalize(self.body(x).mean(dim=(2, 3)), dim=1)

model = Embedder().to(dev).eval()
# prétraitement unique pour toutes les images (galeries et requêtes) : plus petit côté 256, recadrage central 256x256
prep = T.Compose([T.Resize(256), T.CenterCrop(256), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
def load_img(p): return prep(Image.open(p).convert('RGB'))

def embed(paths, desc='embeddings'):
    feats = []
    with ThreadPoolExecutor(N_THREADS) as ex:
        for i in tqdm(range(0, len(paths), BATCH), desc=desc):
            x = torch.stack(list(ex.map(load_img, paths[i:i + BATCH]))).to(dev)
            feats.append(model(x).cpu().numpy())
    return np.concatenate(feats).astype('float32')

In [ ]:
# 5) DTD : embeddings des 5640 images + évaluation sur le split officiel 1
img_root, lab_root = DTD_DIR / 'images', DTD_DIR / 'labels'
cats = sorted(d.name for d in img_root.iterdir() if d.is_dir())
dtd_rel = sorted(f'{c}/{f.name}' for c in cats for f in (img_root / c).iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'})
dtd_lab = np.array([r.split('/')[0] for r in dtd_rel]); pos = {r: i for i, r in enumerate(dtd_rel)}
print(len(dtd_rel), 'images DTD,', len(cats), 'catégories')
t0 = time.time(); X_dtd = embed([img_root / r for r in dtd_rel], 'DTD'); print(f'{time.time() - t0:.0f}s | forme {X_dtd.shape}')

def split_idx(name):
    return np.array([pos[l.strip()] for l in open(lab_root / f'{name}1.txt') if l.strip() in pos])
g_idx = np.r_[split_idx('train'), split_idx('val')]; q_idx = split_idx('test')
met, order, rel = retrieval_metrics(X_dtd[q_idx], X_dtd[g_idx], dtd_lab[q_idx], dtd_lab[g_idx])
chance = float(np.mean([(dtd_lab[g_idx] == c).mean() for c in dtd_lab[q_idx]]))
print(f'DTD split 1 : galerie {len(g_idx)} images, {len(q_idx)} requêtes')
print({k: round(v, 3) for k, v in met.items()}, '| P@1 au hasard :', round(chance, 3))

top1 = dtd_lab[g_idx][order[:, 0]]; ok = top1 == dtd_lab[q_idx]
per = pd.DataFrame({'cat': dtd_lab[q_idx], 'ok': ok}).groupby('cat').ok.mean().sort_values()
print('\nCatégories les plus difficiles (P@1) :'); display(per.head(8).round(2).to_frame('P@1'))
conf = collections.Counter((a, b) for a, b, o in zip(dtd_lab[q_idx], top1, ok) if not o).most_common(6)
print('Confusions les plus fréquentes (vraie -> voisine) :', conf)
DTD_EVAL = dict(split='officiel 1 (galerie=train+val, requêtes=test)', n_gallery=int(len(g_idx)), n_queries=int(len(q_idx)),
                metrics={k: round(v, 4) for k, v in met.items()}, chance_P_at_1=round(chance, 4),
                hardest_categories={k: round(float(v), 3) for k, v in per.head(8).items()}, top_confusions=[[a, b, n] for (a, b), n in conf])

In [ ]:
# 6) Références du domaine WeaveGuard (v2) + évaluation train -> val
if not V2.exists(): zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v2.zip').extractall(V2)
stats = json.load(open(DRIVE_OUT / 'dataset_stats_v2.json')); man = pd.read_csv(V2 / 'manifest.csv')
DS_HASH = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()
assert DS_HASH == stats['dataset_hash_sha256'], 'Le dataset local ne correspond pas au v2 de référence !'
NAMES = stats['classes']; cid = {n: i for i, n in enumerate(NAMES)}
sub = man[man.split.isin(['train', 'val'])].reset_index(drop=True)
X_wg = embed([V2 / 'images' / s / n for s, n in zip(sub.split, sub.name)], 'WeaveGuard v2')
P = np.zeros((len(sub), len(NAMES)), dtype=np.float32)
for i, c in enumerate(sub.classes):
    for nm in str(c).split(';'): P[i, cid[nm]] = 1
tr, va = np.where(sub.split == 'train')[0], np.where(sub.split == 'val')[0]
S = X_wg[va] @ X_wg[tr].T; order_wg = np.argsort(-S, axis=1)
WG_EVAL = {k: round(v, 3) for k, v in multilabel_hits(P[va], P[tr], order_wg).items()}
WG_EVAL.update(n_gallery=int(len(tr)), n_queries=int(len(va)))
print('Domaine WeaveGuard (train -> val) :', WG_EVAL)
print('NB : "hit" = au moins un voisin partage une classe de défaut avec la requête. Mesure de qualité de l\'embedding, PAS une détection.')

In [ ]:
# 7) Index FAISS (produit scalaire sur vecteurs normalisés = similarité cosinus), métadonnées, recherche
idx_dtd = faiss.IndexFlatIP(X_dtd.shape[1]); idx_dtd.add(X_dtd)
meta_dtd = [dict(ref_id=f'dtd:{r}', source='DTD', category=r.split('/')[0]) for r in dtd_rel]
idx_wg = faiss.IndexFlatIP(X_wg.shape[1]); idx_wg.add(X_wg)
meta_wg = [dict(ref_id=f'weaveguard-v2:{n}', source='WeaveGuard v2', split=s,
                classes=[NAMES[j] for j in np.where(P[i] > 0)[0]]) for i, (s, n) in enumerate(zip(sub.split, sub.name))]
INDEXES = dict(dtd=(idx_dtd, meta_dtd, ALLOWED_FIELDS_DTD), weaveguard=(idx_wg, meta_wg, ALLOWED_FIELDS_WG))

def search(img, which='weaveguard', k=5):
    # contrat de sortie cible pour l'API (§10) : métrique explicite, version de l'extracteur, champs autorisés uniquement
    x = prep(img.convert('RGB'))[None].to(dev)
    q = model(x).cpu().numpy().astype('float32')
    index, meta, allowed = INDEXES[which]; D, I = index.search(q, k)
    return dict(engine='similarity', extractor=EXTRACTOR_ID, index=which, metric='cosine_similarity', higher_is_better=True,
                label='Textiles similaires', note='similarité visuelle ; ce n est pas une détection de défaut',
                results=[dict(rank=r + 1, score=round(float(D[0, r]), 4), **{f: meta[int(I[0, r])][f] for f in allowed if f in meta[int(I[0, r])]}) for r in range(k)])

example = Image.open(V2 / 'images' / 'val' / sorted(man[man.split == 'val'].name)[0])
print(json.dumps(search(example, 'weaveguard', 3), indent=1, ensure_ascii=False))

In [ ]:
# 8) Vérification visuelle : 6 images de validation et leurs voisins (domaine WeaveGuard, puis DTD)
import matplotlib.pyplot as plt
vnames = sorted(man[man.split == 'val'].name); sample = random.Random(SEED).sample(vnames, 6)
def thumb(path, size=220):
    im = Image.open(path).convert('RGB'); im.thumbnail((size, size)); return im
for which in ('weaveguard', 'dtd'):
    fig, axes = plt.subplots(6, 5, figsize=(14, 17))
    for r, n in enumerate(sample):
        q = Image.open(V2 / 'images' / 'val' / n); res = search(q, which, 4)
        qcls = ';'.join(NAMES[j] for j in np.where(P[sub.index[sub.name == n][0]] > 0)[0])
        axes[r, 0].imshow(thumb(V2 / 'images' / 'val' / n)); axes[r, 0].set_title(f'REQUÊTE\n{qcls}', fontsize=8)
        for c, h in enumerate(res['results']):
            if which == 'weaveguard':
                s_, nm = h['ref_id'].split(':')[1], None
                path = V2 / 'images' / h['split'] / h['ref_id'].split(':', 1)[1]; title = f"{h['score']:.3f}\n{';'.join(h['classes'])}"
            else:
                path = img_root / h['ref_id'].split(':', 1)[1]; title = f"{h['score']:.3f}\n{h['category']}"
            axes[r, c + 1].imshow(thumb(path)); axes[r, c + 1].set_title(title, fontsize=8)
        for ax in axes[r]: ax.axis('off')
    plt.suptitle(f'Textiles similaires : index {which} (similarité cosinus ; visuelle, pas une détection)'); plt.tight_layout(); plt.show()

In [ ]:
# 9) Latence (Colab) : extraction + recherche
def t_ms(fn, n=20):
    fn()
    if dev.type == 'cuda': torch.cuda.synchronize()
    t = time.time()
    for _ in range(n): fn()
    if dev.type == 'cuda': torch.cuda.synchronize()
    return (time.time() - t) / n * 1000
x1 = prep(example.convert('RGB'))[None].to(dev); q1 = model(x1).cpu().numpy().astype('float32')
LAT = dict(embedding_ms=round(t_ms(lambda: model(x1)), 1), search_dtd_ms=round(t_ms(lambda: idx_dtd.search(q1, K)), 2),
           search_weaveguard_ms=round(t_ms(lambda: idx_wg.search(q1, K)), 2), device=torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu')
print(LAT)

In [ ]:
# 10) Export versionné + fiche modèle (rapport à coller dans le chat)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
faiss.write_index(idx_dtd, str(OUT / 'index_dtd.faiss')); faiss.write_index(idx_wg, str(OUT / 'index_weaveguard.faiss'))
json.dump(meta_dtd, open(OUT / 'metadata_dtd.json', 'w'), ensure_ascii=False); json.dump(meta_wg, open(OUT / 'metadata_weaveguard.json', 'w'), ensure_ascii=False)
config = dict(extractor_id=EXTRACTOR_ID, backbone='resnet50', weights='ImageNet IMAGENET1K_V1', pooling='moyenne globale layer4', dim=int(X_dtd.shape[1]),
              normalization='L2', preprocess='Resize(256) plus petit côté + CenterCrop(256) + ImageNet mean/std', index_type='faiss.IndexFlatIP (recherche exacte)',
              metric='cosine_similarity', higher_is_better=True, k_default=5, allowed_fields=dict(dtd=ALLOWED_FIELDS_DTD, weaveguard=ALLOWED_FIELDS_WG))
json.dump(config, open(OUT / 'config.json', 'w'), indent=1, ensure_ascii=False)
card = dict(model_id='similarity_v1', role='Recherche de textiles similaires (similarité visuelle, pas une détection de défaut)', config=config,
            artifacts_sha256={f: sha(OUT / f) for f in ('index_dtd.faiss', 'index_weaveguard.faiss', 'metadata_dtd.json', 'metadata_weaveguard.json', 'config.json')},
            galleries=dict(dtd=dict(n=len(meta_dtd), note='toutes les images DTD'),
                           weaveguard=dict(n=len(meta_wg), dataset_version='v2', dataset_hash=DS_HASH, note='images train+val ; le test reste hors index')),
            evaluation=dict(dtd=DTD_EVAL, weaveguard_domain=WG_EVAL), latency_colab=LAT,
            ui_label='Textiles similaires', never='ne pas présenter comme un défaut ni comme un pourcentage de confiance',
            limits=['DTD = 47 catégories génériques de textures, pas des tissus industriels',
                    'index évalué sur DTD split 1 (galerie train+val) ; l index exporté contient les 5640 images',
                    'la mesure du domaine WeaveGuard est un accord de classe entre voisins, pas une vérité de similarité',
                    'prétraitement à recadrage central : les bords des grandes images sont ignorés',
                    'latence à remesurer sur le matériel cible (GTX 1050)'],
            versions=dict(torch=torch.__version__, torchvision=torchvision.__version__, faiss=faiss.__version__),
            created=datetime.datetime.now().isoformat(timespec='seconds'))
(OUT / 'model_card.json').write_text(json.dumps(card, indent=1, ensure_ascii=False), encoding='utf-8')
print('Exporté :', OUT); print('=' * 70)
print(json.dumps(dict(dtd=DTD_EVAL, weaveguard_domain=WG_EVAL, latency=LAT, galleries=card['galleries'], index_sizes_mb={f: round((OUT / f).stat().st_size / 1e6, 1) for f in ('index_dtd.faiss', 'index_weaveguard.faiss')}), indent=1, ensure_ascii=False))